In [0]:
# Northwind Cross-Cloud Ingestion: Azure SQL (Azure) -> Databricks (AWS)
# Credentials stored in Databricks Secrets -- never hardcoded

jdbc_url = (
    "jdbc:sqlserver://northwind-directionaldatallc.database.windows.net:1433;"
    "database=Northwind;encrypt=true;trustServerCertificate=false;loginTimeout=30;"
)

username = dbutils.secrets.get(scope="azure-sql", key="username")
password = dbutils.secrets.get(scope="azure-sql", key="password")

tables = [
    "Orders", "Order Details", "Customers", "Products",
    "Categories", "Suppliers", "Employees",
    "EmployeeTerritories", "Territories", "Region", "Shippers"
]

for table in tables:
    df = (spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", f"dbo.[{table}]")
        .option("user", username)
        .option("password", password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    safe_name = table.lower().replace(" ", "_")
    target = f"northwind.raw.{safe_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(target)
    print(f"OK  {table:25s} -> {target}  ({df.count()} rows)")

print("Ingestion complete.")
